# Chapter 4 &mdash; Formal Acceptance: $\delta$, $\hat{\delta}$, `accepts`

**Concept 10 of the Chapter 4 decomposition:** *Formal Acceptance: $\delta$, $\hat{\delta}$, and the `accepts` Predicate*

One function per layer: step a symbol, run a string, run from any state, decide.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Delta-Hat-Acceptance/Concept-Delta-Hat-Acceptance.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Acceptance is defined by three functions, and Jove has one per layer:

* **$\delta$ = `step_dfa(D,q,c)`** &mdash; one state, one symbol, one next state;
* **$\hat{\delta}$ = `run_dfa_h(D,s,q)`** &mdash; a *string* from an *arbitrary* state,
  defined recursively with basis $\hat{\delta}(D,q,\varepsilon)=q$;
* **`run_dfa(D,s)`** &mdash; $\hat{\delta}$ from $q_0$;
* **`accepts_dfa(D,s)`** &mdash; run, then test membership in $F$.

The `_h` suffix means *helper* and marks the general-starting-state version.

## 2. Definitions

### The machine

In [ ]:
D = md2mc('''DFA
I : 0 -> I
I : 1 -> F
F : 0 -> F
F : 1 -> I
''')

### $\hat{\delta}$, written out

The recursion is on the **string**, not the machine.

In [ ]:
def delta_hat(D, q, s):
    return q if s == '' else delta_hat(D, step_dfa(D, q, s[0]), s[1:])

<!-- nav-strip -->

---

&larr;&nbsp;[Ch4&nbsp;9.&nbsp;Basics of Designing a DFA](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Designing-A-DFA/Concept-Designing-A-DFA.ipynb) &nbsp;&middot;&nbsp; [**Chapter 4** index](https://github.com/ganeshutah/Jove/blob/master/Chapter4-DFA/README.md) &nbsp;&middot;&nbsp; [Ch4&nbsp;11.&nbsp;How to Read the Jove Code](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Reading-Jove-Code/Concept-Reading-Jove-Code.ipynb)&nbsp;&rarr;

---

## 3. Tests

The basis case $\hat{\delta}(D,q,\varepsilon)=q$ &mdash; which is why an initial-and-final state accepts $\varepsilon$.

In [ ]:
for q in sorted(D["Q"]):
    print("delta_hat(D, %s, '') = %s" % (q, delta_hat(D, q, '')))
assert all(delta_hat(D, q, '') == q for q in D["Q"])

Our recursion agrees with Jove's `run_dfa` and `run_dfa_h`.

In [ ]:
for s in ['', '1', '10', '111', '1010']:
    mine, jove = delta_hat(D, D["q0"], s), run_dfa(D, s)
    print("s=%-7r delta_hat=%-3s run_dfa=%-3s agree=%s" % (s, mine, jove, mine == jove))
assert all(delta_hat(D, D["q0"], s) == run_dfa(D, s)
           for s in ['', '1', '10', '111', '1010', '0000'])

`run_dfa_h` starts anywhere &mdash; useful for reasoning mid-computation.

In [ ]:
print("from F on '1'  ->", run_dfa_h(D, '1', 'F'))
print("from I on '11' ->", run_dfa_h(D, '11', 'I'))
assert run_dfa_h(D, '', 'F') == 'F'

And acceptance is just membership in $F$.

In [ ]:
for s in ['1', '11', '101']:
    print("%-6r ends in %-3s in F? %-5s accepts_dfa=%s"
          % (s, run_dfa(D, s), run_dfa(D, s) in D["F"], accepts_dfa(D, s)))
assert all((run_dfa(D, s) in D["F"]) == accepts_dfa(D, s) for s in ['1','11','101',''])

## 4. Exercises


1. Rewrite `delta_hat` to recurse on the *last* symbol instead of the first.
   Does it still agree?
2. What is $\hat{\delta}(D,q,s)$ when $s$ has 1000 symbols? Try it &mdash; and explain
   the error.
3. Why does the definition need `run_dfa_h` at all, when `run_dfa` exists?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 254 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter4-DFA/Concept-Delta-Hat-Acceptance')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')